# 第76章 旭日图（px.sunburst）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 13 / 18 步：表达层级、流程、贡献与地域**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 矩形树图（px.treemap）  →  **本章任务：** 旭日图（px.sunburst）  →  **下一步：** 漏斗图（px.funnel）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

做饼图、漏斗图只能看到一层占比，一旦数据带上父子层级，比如“部门”下面还有“品类”，平面图表就装不下了。


## 本章目标

学完本章，你将能够：

- **理解**：理解「旭日图（px.sunburst）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「旭日图（px.sunburst）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「旭日图（px.sunburst）」并读出其中的结论。


## 适用场景

**背景引入**：做饼图、漏斗图只能看到一层占比，一旦数据带上父子层级，比如“部门”下面还有“品类”，平面图表就装不下了。旭日图用一圈圈同心环把父级放在内圈、子级放在外圈，既能看到整体构成，又能逐层向下钻取，很适合梳理“总—分”式的分类结构。看懂它，面对部门、区域、渠道这类层级数据，就能一图说清楚“谁占了大头、又是怎么分下去的”。（可以把它想成“靶子”：最里面的靶心是总体的头（父级），往外一圈圈是往下分的层级（子级）；每环被切成一块块同心扇形，扇形角度越大占比越高，点一下还能顺着层级钻进去。）

层级较浅，需要同时呈现父级和子级构成。


## 数据结构

层级路径及非负数值。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 添加 maxdepth=2 参数，观察限制显示深度对多层旭日图的影响
2. 修改 textinfo 从 "label+percent parent" 为 "label+percent entry"，对比不同占比参考基准
3. 将 color 从分类字段改为数值指标（如 color="sales"），说明连续色阶对规模的编码作用


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.sunburst()`、`fig.update_traces()`、`fig.show()` | 层级较浅，需要同时呈现父级和子级构成。 | 层级和节点过多 |
| 进阶变体 | `orders.groupby()`、`px.sunburst()`、`fig.show()`、`.sum()` | 在基础图表上增加分组、注释、布局或交互 | 跨父级直接比较外环角度 |
| 关键参数 | `path` | 层级 | 层级和节点过多 |
| 关键参数 | `values` | 扇区大小 | 跨父级直接比较外环角度 |
| 关键参数 | `maxdepth` | 显示深度 | 颜色与层级关系不清 |
| 关键参数 | `color` | 颜色 | 层级和节点过多 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-76 -->
### 数学推导｜构成图的守恒关系

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先定义同一总体。** $T=\sum_{i=1}^{K}x_i$。

**第 2 步｜每个类别除以同一总体。** $s_i=x_i/T$。

**第 3 步｜验证守恒。** $\sum_i s_i=\sum_i x_i/T=T/T=1$。层级图还要逐个父节点检查

$$
x_{parent}=\sum_{c\in children(parent)}x_c
$$

否则面积虽然能画出来，却不再代表一致的层级构成。

**把上面的关系收束为本章计算式：**

$$
s_i=\frac{x_i}{\sum_jx_j},\qquad \sum_i s_i=1
$$

**符号解释：** $s_i$ 是类别或节点占总体的比例。

**代码对应：** 先聚合并检查 `share.sum()` 接近 1，再传给饼图、矩形树图或旭日图。

**使用边界：** 层级图要求父节点值与子节点口径一致；类别过多时应合并长尾。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.sunburst(
    hierarchy,
    path=["department", "category"],
    values="sales",
    color="department",
    title="部门与品类销售层级",
)
fig.update_traces(textinfo="label+percent parent")
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：下面还是用同一份 `hierarchy` 数据画旭日图，但请改动两个参数观察变化：一是给 `px.sunburst(...)` 增加 `maxdepth=2`，看显示深度被限制到两层后，哪些更深的子类不再展开；二是把 `textinfo` 从 `"label+percent parent"` 改成 `"label+percent entry"`，看扇区上的占比以谁的体量为基准。对照运行前后的图形，用一句话写下你看到的差异。


In [ ]:
try:
    pass
    # 请在下方填写代码：用 hierarchy 画一张旭日图，并改动两个参数。
    # 提示：
    #   1. fig = px.sunburst(hierarchy, path=["department", "category"], values="sales", ...)
    #   2. 在 px.sunburst 里增加 maxdepth=2
    #   3. fig.update_traces(textinfo="label+percent entry")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
sun = orders.groupby(["region", "channel", "category"], as_index=False)[
    "sales"
].sum()
fig = px.sunburst(
    sun,
    path=["region", "channel", "category"],
    values="sales",
    color="region",
    maxdepth=3,
    title="区域、渠道与品类结构",
)
fig.show()


## 参数说明

- path：层级
- values：扇区大小
- maxdepth：显示深度
- color：颜色


## 结果解读

内环是上级，外环是下级；角度表示相对规模。


## 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 层级和节点过多
- 跨父级直接比较外环角度
- 颜色与层级关系不清


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：用「销售额」观察层级结构的变化
    # 【目标】换 values 列，练习从不同口径看层级占比。
    import plotly.express as px

    # 起点示例(已可运行)：values 用 sales，观察各层级贡献。
    fig = px.sunburst(
        hierarchy,
        path=["department", "category"],
        values="sales",
        color="department",
        title="部门与品类销售层级",
    )
    fig.update_traces(textinfo="label+percent parent")
    fig.show()

    # ---- 反思记录：内圈与外圈分别代表什么层级 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用同心环展示层级路径和各层占比，并支持点击下钻。


### 你已经掌握

- 判断旭日图（px.sunburst）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `path` | 层级 |
| `values` | 扇区大小 |
| `maxdepth` | 显示深度 |
| `color` | 颜色 |


### 需要注意

- 层级和节点过多
- 跨父级直接比较外环角度
- 颜色与层级关系不清


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案
fig = px.sunburst(
    hierarchy,
    path=["department", "category"],
    values="sales",
    maxdepth=2,  # 只显示两层；更深的子类被折叠
    color="department",
)
fig.update_traces(textinfo="label+percent entry")


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice_sun = orders.groupby(["category", "region"], as_index=False)[
    "sales"
].sum()
fig = px.sunburst(
    practice_sun,
    path=[px.Constant("全部品类"), "category", "region"],
    values="sales",
    color="category",
    title="品类与区域销售层级",
)
fig.show()
